# Head-direction tuning with pynapple: Object at position 1 vs no object

Compares each matched unit's head-direction tuning curve between two conditions of one session:
- **object**: an `Object` registered at position `1` is in the arena;
- **no object**: the arena is empty, no `Object` and no `Shelter` (by default only the time before the first
  registration of either).
Optionally (`OBJECT_ALONE = True`) the object condition is limited to times when nothing else (e.g. a shelter) is
present.

Follows the pynapple head-direction tutorial (https://pynapple.org/examples/tutorial_HD_dataset.html):
`compute_tuning_curves` → `compute_mutual_information` → smoothed polar plots.

**Requirements** (in the repo you run this from): `datajoint`, `aeon_mecha` (imported only to register the
`<xarray>` / `<blob@dj_store>` codecs), `pynapple >= 0.9`, `xarray`, `scipy`, `matplotlib`, and a `datajoint.json`
at the repo root with the `aeondj` credentials and the `dj_store` store (the first cell changes into that folder, as
the other notebooks do). Tables are read by schema name, so the AEON_OVC code is not needed.

**How the variables were computed** (AEON_OVC `src/dj_pipeline/kinematics.py`, table `ovc_behav.Kinematics`):
- SLEAP keypoints (legacy SLEAP 1.4 model) → foragingABC pose QC (low-confidence points removed, jumps flagged,
  gaps ≤ 3 frames interpolated) → mapped to arena cm with the CameraTop calibration (undistort + homography,
  origin bottom-left, y up).
- `head_direction`: perpendicular to the left→right ear axis, pointing towards the nose (sign checked against
  nose/head keypoints), circular Savitzky–Golay smoothing over 11 frames (0.22 s). Degrees, **0° = +x (arena right),
  counter-clockwise**, −180..180.
- `head_direction_confidence`: ear separation / its median (0–1); halved when the sign could only be checked against
  spine1, 0 when an ear is missing.
- `speed`: centroid velocity (gradient, Gaussian σ = 5 frames), cm/s.
- Frames are 50 Hz on the HARP clock, the same clock as the spike times (`spike_sorting.UnitMatching.Spikes`).
- Object presence (`ovc_behav.ObjectRegistration.Object`): from the registration frame until 5 min before the next
  registration (continues without a gap if the next registration has the same identity and position).
- Light (`ovc_behav.LightPeriod.Period`): from Environment LightEvents channels 3/7 only; `dark` = light value 0.

In [ ]:
import os
from pathlib import Path

# Run from the repo root so datajoint.json is picked up (aeon.dj_pipeline reads it on import)
root = Path.cwd()
while not (root / "datajoint.json").exists() and root != root.parent:
    root = root.parent
if not (root / "datajoint.json").exists():
    raise FileNotFoundError("No datajoint.json in this folder or any parent; start Jupyter inside the repo")
os.chdir(root)
print("Working directory:", root)

import datajoint as dj
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from scipy.ndimage import gaussian_filter1d

try:
    import pynapple as nap
except ImportError as err:
    raise ImportError("pynapple is not installed in this kernel: `uv add 'pynapple>=0.9'` (or pip install)") from err
assert hasattr(nap, "compute_tuning_curves"), f"needs pynapple >= 0.9, found {nap.__version__}"

import aeon.dj_pipeline  # noqa: F401  registers the <xarray> / <blob@dj_store> codecs used below

EXPERIMENT = "ProjectAeonOVC-2026-09-14_001"
SCHEMA_PREFIX = "aeon_foragingOVC_"
CAMERA = "CameraTop"

OBJECT_IDENTITY, OBJECT_POSITION = "Object", "1"
OBJECT_ALONE = False         # True: object condition only while no other object/shelter is present
NO_OBJECT = "before_first"   # empty arena (no Object, no Shelter). "before_first": before the first registration;
                             # "all": any empty time, which includes the 5-min removal gaps between registrations
LIGHT = None                 # None = all; "light" or "dark" to keep only that light condition
MIN_SPEED = None             # cm/s; e.g. 2 to drop immobility (None = keep all)
MIN_HD_CONFIDENCE = 0.5      # drop frames with an unreliable ear axis

LABELS = ["good"]            # bombcell labels of the global units kept: good if good in any of its blocks, else its
                             # most common label. LABELS = None keeps every matched unit (no bombcell needed)
BINS = 60                    # 6 deg bins
SMOOTH_SIGMA = 2             # bins, circular smoothing for plotting only
MIN_TIME_S = 300             # min analysed time per unit and condition
N_PLOT = 20                  # units shown in the polar grid

behav = dj.VirtualModule("ovc_behav", SCHEMA_PREFIX + "ovc_behav")
ss = dj.VirtualModule("spike_sorting", SCHEMA_PREFIX + "spike_sorting")
bombcell = dj.VirtualModule("ovc_bombcell", SCHEMA_PREFIX + "ovc_bombcell") if LABELS else None
exp = {"experiment_name": EXPERIMENT}

## Helpers

pynapple works in float seconds; every HARP datetime is converted relative to `REF`, the session's first
kinematics chunk.

In [ ]:
chunk_starts = np.sort((behav.Kinematics & exp & {"device_name": CAMERA}).to_arrays("chunk_start"))
assert len(chunk_starts), f"No Kinematics for {EXPERIMENT}: run populate_ovc_behaviour.py first"
REF = np.datetime64(pd.Timestamp(chunk_starts[0]), "ns")


def to_s(t):
    """HARP datetime(s) -> seconds since REF."""
    return (np.asarray(t, dtype="datetime64[ns]") - REF) / np.timedelta64(1, "s")


def intervals(starts, ends):
    """IntervalSet from datetime starts/ends (merged if they overlap)."""
    if len(starts) == 0:
        return nap.IntervalSet(start=[], end=[])
    s, e = to_s(starts), to_s(ends)
    order = np.argsort(s)
    s, e = s[order], e[order]
    merged = [[s[0], e[0]]]
    for a, b in zip(s[1:], e[1:]):
        if a <= merged[-1][1]:
            merged[-1][1] = max(merged[-1][1], b)
        else:
            merged.append([a, b])
    merged = np.array(merged)
    return nap.IntervalSet(start=merged[:, 0], end=merged[:, 1])


def mask_to_intervals(t, mask, max_gap=0.1):
    """IntervalSet of runs of consecutive samples where mask is True; a run ends at a masked-out sample
    or a time gap > max_gap s (at least 1.5 frame intervals). Each run lasts until its last sample + 1 frame."""
    t, mask = np.asarray(t), np.asarray(mask, dtype=bool)
    if not mask.any():
        return nap.IntervalSet(start=[], end=[])
    dt = float(np.median(np.diff(t))) if len(t) > 1 else 0.0
    max_gap = max(max_gap, 1.5 * dt)
    idx = np.flatnonzero(mask)
    breaks = np.flatnonzero((np.diff(idx) > 1) | (np.diff(t[idx]) > max_gap))
    starts = idx[np.r_[0, breaks + 1]]
    ends = idx[np.r_[breaks, len(idx) - 1]]
    return nap.IntervalSet(start=t[starts], end=t[ends] + dt)

## Conditions: Object at position 1 vs empty arena

In [ ]:
objects = pd.DataFrame(
    (behav.ObjectRegistration.Object & exp).proj("identity", "position_category", "object_onset", "object_offset")
    .to_dicts(order_by="object_id")
)
tracked_end = np.datetime64(pd.Timestamp(chunk_starts[-1]) + pd.Timedelta(hours=1), "ns")
objects["end"] = objects["object_offset"].fillna(pd.Timestamp(tracked_end))
is_target = (objects["identity"] == OBJECT_IDENTITY) & (objects["position_category"] == OBJECT_POSITION)
target = objects[is_target]
print(objects[["object_id", "identity", "position_category", "object_onset", "object_offset"]].to_string(index=False))

tracked = nap.IntervalSet(start=0.0, end=to_s(tracked_end))
anything = intervals(objects["object_onset"], objects["end"])                      # any object or shelter present
others = intervals(objects.loc[~is_target, "object_onset"], objects.loc[~is_target, "end"])
condition_ep = {
    "object": intervals(target["object_onset"], target["end"]),
    "no object": tracked.set_diff(anything) if len(anything) else tracked,      # empty arena
}
if OBJECT_ALONE and len(others):
    condition_ep["object"] = condition_ep["object"].set_diff(others)
if NO_OBJECT == "before_first" and len(objects):
    first_registration = objects["object_onset"].min()
    condition_ep["no object"] = condition_ep["no object"].intersect(
        nap.IntervalSet(start=0.0, end=to_s(first_registration))
    )

if LIGHT is not None:
    periods = pd.DataFrame((behav.LightPeriod.Period & exp & {"condition": LIGHT}).to_dicts())
    light_ep = intervals(periods["period_start"], periods["period_end"])
    condition_ep = {c: ep.intersect(light_ep) for c, ep in condition_ep.items()}

for c, ep in condition_ep.items():
    print(f"{c:10s}: {len(ep)} interval(s), {ep.tot_length() / 3600:.2f} h")

## Head direction (only the kinematics chunks overlapping either condition)

In [ ]:
span = condition_ep["object"].union(condition_ep["no object"])
lo = REF + np.timedelta64(int(span.start.min() * 1e9), "ns") - np.timedelta64(1, "h")
hi = REF + np.timedelta64(int(span.end.max() * 1e9), "ns")
rows = (
    behav.Kinematics & exp & {"device_name": CAMERA}
    & f"chunk_start >= '{pd.Timestamp(lo)}'" & f"chunk_start <= '{pd.Timestamp(hi)}'"
).to_dicts(order_by="chunk_start")

parts = []
for r in rows:
    k = r["kinematics_data"]
    parts.append(pd.DataFrame({
        "t": to_s(k["time"].values),
        "hd": k["head_direction"].values,
        "conf": k["head_direction_confidence"].values,
        "speed": k["speed"].values,
    }))
kin = pd.concat(parts).drop_duplicates("t").sort_values("t").reset_index(drop=True)
print(f"{len(rows)} chunks, {len(kin)} frames")

valid = kin["hd"].notna() & (kin["conf"] >= MIN_HD_CONFIDENCE)
if MIN_SPEED is not None:
    valid &= kin["speed"] >= MIN_SPEED
valid_ep = mask_to_intervals(kin["t"].values, valid.values)

# pynapple expects radians; tuning range 0..2pi (0 = arena +x, counter-clockwise)
head_direction = nap.Tsd(t=kin["t"].values, d=np.deg2rad(kin["hd"].values) % (2 * np.pi))
analysis_ep = {c: ep.intersect(valid_ep) for c, ep in condition_ep.items()}
for c, ep in analysis_ep.items():
    print(f"{c:10s}: {ep.tot_length() / 3600:.2f} h with valid head direction")

## Spikes of matched units

`UnitMatching.Spikes` holds each global unit's spikes per ephys chunk, each chunk taken from one block (overlaps
counted once). A global unit only exists during the blocks it was matched in, so each unit's tuning is computed
only over the time its blocks cover (`unit_ep`); otherwise silent time outside its blocks would lower its rates.

In [ ]:
units = pd.DataFrame((ss.UnitMatching.Unit & exp).proj("global_unit").to_dicts())
if LABELS:
    labels = pd.DataFrame((bombcell.BombcellLabel.Unit & exp).proj("bombcell_label").to_dicts())
    on = [c for c in units.columns if c in labels.columns]
    units = units.merge(labels, on=on, how="left")
    # global label: good if bombcell labels the unit good in any of its blocks, else its most common label
    global_label = units.groupby(["insertion_number", "global_unit"])["bombcell_label"].agg(
        lambda s: "good" if (s == "good").any() else (s.mode().iloc[0] if s.notna().any() else None)
    )
    keep = global_label[global_label.isin(LABELS)]
    units = units.set_index(["insertion_number", "global_unit"]).loc[keep.index].reset_index()
units["uid"] = units["insertion_number"] * 100000 + units["global_unit"]  # TsGroup needs int keys
print(f"{units['uid'].nunique()} global units kept")

# each unit's time coverage = union of the blocks it was matched in
unit_ep = {uid: intervals(g["block_start"], g["block_end"]) for uid, g in units.groupby("uid")}
shank = units.groupby("uid")["electrode_group"].first()

spike_rows = (
    ss.UnitMatching.Spikes & exp
    & f"chunk_start >= '{pd.Timestamp(lo)}'" & f"chunk_start <= '{pd.Timestamp(hi)}'"
).proj("spike_times").to_dicts()
times = {}
for r in spike_rows:
    uid = r["insertion_number"] * 100000 + r["global_unit"]
    if uid in unit_ep:
        times.setdefault(uid, []).append(to_s(r["spike_times"]))
spikes = nap.TsGroup({uid: nap.Ts(t=np.sort(np.concatenate(t))) for uid, t in times.items()})
spikes.set_info(shank=shank.reindex(list(spikes.keys())).values)
spikes

## Tuning curves per unit and condition

One `compute_tuning_curves` call per unit, restricted to (condition ∩ valid head direction ∩ the unit's blocks), so
both occupancy and spike counts come from exactly the same time. Units with less than `MIN_TIME_S` in a condition
are skipped for that condition.

In [ ]:
def mean_vector_length(tc_row, angles):
    """Rayleigh vector length of a tuning curve (0 = flat, 1 = single direction)."""
    r = np.nan_to_num(tc_row)
    return float(np.abs((r * np.exp(1j * angles)).sum()) / r.sum()) if r.sum() > 0 else np.nan


curves, stats = {c: {} for c in analysis_ep}, []
for uid in spikes.keys():
    for c, ep in analysis_ep.items():
        ep_u = ep.intersect(unit_ep[uid])
        if ep_u.tot_length() < MIN_TIME_S:
            continue
        tc = nap.compute_tuning_curves(
            data=spikes[[uid]], features=head_direction, bins=BINS, range=(0, 2 * np.pi),
            epochs=ep_u, feature_names=["head_direction"],
        )
        mi = nap.compute_mutual_information(tc)
        angles = tc["head_direction"].values
        row = tc.sel(unit=uid).values
        smooth = gaussian_filter1d(np.nan_to_num(row), SMOOTH_SIGMA, mode="wrap")
        curves[c][uid] = row
        stats.append({
            "uid": uid, "condition": c, "shank": spikes[uid].shank if hasattr(spikes[uid], "shank") else shank[uid],
            "time_h": ep_u.tot_length() / 3600,
            "mean_rate": float(np.nanmean(row)),
            "mi_bits_per_s": float(mi.iloc[0, 0]),
            "mean_vector_length": mean_vector_length(row, angles),
            "preferred_deg": float(np.degrees(angles[np.argmax(smooth)])),
        })
ANGLES = angles
stats = pd.DataFrame(stats)
wide = stats.pivot(index="uid", columns="condition", values=["mi_bits_per_s", "mean_vector_length", "preferred_deg"])
wide.columns = [f"{v}_{c.replace(' ', '_')}" for v, c in wide.columns]
wide = wide.join(shank.rename("shank"))
wide.sort_values("mi_bits_per_s_no_object", ascending=False).round(3).head(20)

## Polar tuning curves (top units by mutual information in either condition)

Grey = empty arena (no object, no shelter), colour = object at position 1. Angles: 0° = arena +x (right), counter-clockwise; the same frame as
the arena plots. Curves are smoothed circularly for display only.

In [ ]:
score = stats.groupby("uid")["mi_bits_per_s"].max().sort_values(ascending=False)
top = [u for u in score.index if all(u in curves[c] for c in curves)][:N_PLOT]
ncol = 5
nrow = int(np.ceil(len(top) / ncol))
fig, axes = plt.subplots(nrow, ncol, subplot_kw={"projection": "polar"}, figsize=(3 * ncol, 3.2 * nrow), squeeze=False)
theta = np.r_[ANGLES, ANGLES[0]]
for ax, uid in zip(axes.ravel(), top):
    for c, colour in (("no object", "0.5"), ("object", "C3")):
        r = gaussian_filter1d(np.nan_to_num(curves[c][uid]), SMOOTH_SIGMA, mode="wrap")
        ax.plot(theta, np.r_[r, r[0]], color=colour, lw=1.5, label=c)
    w = wide.loc[uid]
    ax.set_title(f"{uid} ({shank[uid]})\nR {w['mean_vector_length_no_object']:.2f} / {w['mean_vector_length_object']:.2f}",
                 fontsize=8)
    ax.set_xticks(np.deg2rad([0, 90, 180, 270]))
    ax.set_xticklabels(["0", "90", "180", "270"], fontsize=7)
    ax.tick_params(axis="y", labelsize=6)
for ax in axes.ravel()[len(top):]:
    ax.set_visible(False)
axes.ravel()[0].legend(fontsize=7, loc="lower left", bbox_to_anchor=(-0.3, -0.35))
fig.suptitle(f"{EXPERIMENT}: head-direction tuning (Hz), R = mean vector length (no object / object)", y=1.01)
fig.tight_layout()
plt.show()

## Population comparison

Left: head-direction occupancy per condition (uneven sampling of directions is accounted for by the tuning curves,
but very sparse bins are noisy). Middle: preferred direction in the two conditions for units with a clear tuning
(R ≥ 0.2 in both). Right: mutual information.

In [ ]:
fig = plt.figure(figsize=(14, 4.2))
ax0 = fig.add_subplot(131, projection="polar")
for c, colour in (("no object", "0.5"), ("object", "C3")):
    hd_c = head_direction.restrict(analysis_ep[c]).values
    h, edges = np.histogram(hd_c, bins=BINS, range=(0, 2 * np.pi))
    occ = h / h.sum()
    centres = (edges[:-1] + edges[1:]) / 2
    ax0.plot(np.r_[centres, centres[0]], np.r_[occ, occ[0]], color=colour, label=c)
ax0.set_yticklabels([])
ax0.set_title("occupancy (fraction of frames)", fontsize=9)
ax0.legend(fontsize=7, loc="lower left", bbox_to_anchor=(-0.2, -0.2))

both = wide.dropna(subset=["mean_vector_length_no_object", "mean_vector_length_object"])
tuned = both[(both["mean_vector_length_no_object"] >= 0.2) & (both["mean_vector_length_object"] >= 0.2)]
ax1 = fig.add_subplot(132)
ax1.scatter(tuned["preferred_deg_no_object"], tuned["preferred_deg_object"], s=15)
ax1.plot([0, 360], [0, 360], color="0.7", lw=1)
ax1.set(xlim=(0, 360), ylim=(0, 360), xlabel="preferred HD, no object (deg)", ylabel="preferred HD, object (deg)",
        title=f"{len(tuned)} tuned units (R >= 0.2 in both)")
shift = (tuned["preferred_deg_object"] - tuned["preferred_deg_no_object"] + 180) % 360 - 180
print(f"median |shift in preferred direction|: {shift.abs().median():.1f} deg (n = {len(shift)})")

ax2 = fig.add_subplot(133)
ax2.scatter(both["mi_bits_per_s_no_object"], both["mi_bits_per_s_object"], s=15)
m = np.nanmax(both[["mi_bits_per_s_no_object", "mi_bits_per_s_object"]].values) * 1.05
ax2.plot([0, m], [0, m], color="0.7", lw=1)
ax2.set(xlabel="MI no object (bits/s)", ylabel="MI object (bits/s)", title=f"{len(both)} units in both conditions")
fig.tight_layout()
plt.show()

## Same approach for other variables

Swap `head_direction` for any Tsd built the same way, with a matching `range`:

| Variable | Table / field | Units, range |
|---|---|---|
| egocentric bearing of the object (head) | `ObjectKinematics.object_kinematics["egocentric_bearing_head"]` | deg, −180..180 (0 = ahead, + = left) |
| egocentric bearing vs body / movement | `egocentric_bearing_body`, `egocentric_bearing_movement` | deg, −180..180 |
| allocentric bearing to the object | `allocentric_bearing` | deg, 0 = +x, CCW |
| distance to object centre / nearest edge | `distance_to_centre`, `distance_to_edge` (< 0 inside) | cm |
| approach / retreat | `radial_velocity` (< 0 approach) | cm/s |
| body / movement direction | `Kinematics.kinematics_data["body_direction"]`, `["movement_direction"]` | deg, 0 = +x, CCW |
| angular head velocity, speed | `["angular_head_velocity"]`, `["speed"]` | deg/s, cm/s |
| position (2-D tuning) | `["head_position"]` / `["centroid_position"]` | cm, arena frame |

`ObjectKinematics` rows exist per (chunk, object) only while that object is present; select the rows of the target
objects with `& [{"object_id": i} for i in target["object_id"]]`. The next cell does this for the egocentric bearing
of the object at position 1.

In [ ]:
ok_rows = (
    behav.ObjectKinematics & exp & {"device_name": CAMERA} & [{"object_id": int(i)} for i in target["object_id"]]
).to_dicts(order_by="chunk_start")
ego = pd.concat([
    pd.DataFrame({"t": to_s(r["object_kinematics"]["time"].values),
                  "ego": r["object_kinematics"]["egocentric_bearing_head"].values})
    for r in ok_rows
]).drop_duplicates("t").sort_values("t")
ego = ego[ego["ego"].notna()]
ego_bearing = nap.Tsd(t=ego["t"].values, d=np.deg2rad(ego["ego"].values))  # -pi..pi
ego_ep = analysis_ep["object"].intersect(mask_to_intervals(ego["t"].values, np.ones(len(ego), bool)))

fig, axes = plt.subplots(1, min(5, len(top)), subplot_kw={"projection": "polar"}, figsize=(15, 3.2), squeeze=False)
for ax, uid in zip(axes.ravel(), top):
    ep_u = ego_ep.intersect(unit_ep[uid])
    if ep_u.tot_length() < MIN_TIME_S:
        continue
    tc = nap.compute_tuning_curves(data=spikes[[uid]], features=ego_bearing, bins=BINS, range=(-np.pi, np.pi),
                                   epochs=ep_u, feature_names=["egocentric_bearing"])
    a = tc["egocentric_bearing"].values
    r = gaussian_filter1d(np.nan_to_num(tc.sel(unit=uid).values), SMOOTH_SIGMA, mode="wrap")
    ax.plot(np.r_[a, a[0]], np.r_[r, r[0]], color="C2")
    ax.set_theta_zero_location("N")  # 0 = object straight ahead, + = to the left
    ax.set_xticks(np.deg2rad([0, 90, 180, 270]))
    ax.set_xticklabels(["ahead", "left +90", "behind", "right -90"], fontsize=7)
    ax.tick_params(axis="y", labelsize=6)
    ax.set_title(f"{uid}: object bearing", fontsize=8)
fig.tight_layout()
plt.show()

## Notes

- The head direction comes from the ear axis in the arena cm frame. foragingABC's `pose.HeadingDirection`
  (spine → head centre, image frame, 0 = up, clockwise) uses a different convention; don't mix the two.
- `LABELS` chooses global units by their label across the blocks they appear in: `good` if bombcell labels the unit
  good in any block, otherwise its most common label. All of a kept unit's spikes are used, so firing rates aren't
  lowered in blocks where it carries another label.
- Long gaps (e.g. SLEAP losing the animal inside the shelter) are excluded through `MIN_HD_CONFIDENCE`; check the
  printed valid hours per condition before interpreting differences.
- Compare conditions with matched sampling if needed: restrict both to the same light condition (`LIGHT`) and, for
  `NO_OBJECT = "before_first"`, remember that it is earlier in the session (time-of-day and drift confounds).
- The empty-arena time before the first registration can be short (the overnight shelter is often placed soon after
  the session starts); check the printed hours. `NO_OBJECT = "all"` adds the 5-min removal gaps between
  registrations, when the experimenter may be in the arena.